# Data augmentation, and what it can and cannot fix

Random flips, rotations, and zooms buy ten points here. Understanding why it is not twenty is the more useful half.

**Runs on:** GPU recommended — about 15 minutes on CPU · needs the Kaggle cats-vs-dogs archive &nbsp;·&nbsp; **Slides:** [Chapter 8 — Image Classification](../../../course-web-slides/ch08/index.html) &nbsp;·&nbsp; **Section:** 03 — Using data augmentation

---

## Augmentation as layers

In [ ]:
import keras
from keras import layers

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
])

> **Note** — These are **layers**, active during training and inert at inference — exactly like `Dropout`. Putting them in the model rather than the data pipeline means they follow the model everywhere it goes.

## Seeing what it does

In [ ]:
import matplotlib.pyplot as plt
from keras.utils import image_dataset_from_directory
import pathlib

new_base_dir = pathlib.Path("cats_vs_dogs_small")
train_dataset = image_dataset_from_directory(
    new_base_dir / "train", image_size=(180, 180), batch_size=32)

for images, _ in train_dataset.take(1):
    fig, axes = plt.subplots(3, 6, figsize=(13, 6.6))
    for row in range(3):
        aug = data_augmentation(images)
        for col in range(6):
            axes[row, col].imshow(aug[col].numpy().astype("uint8"))
            axes[row, col].axis("off")
    plt.suptitle("The same six images, three times through augmentation", y=1.0)
    plt.tight_layout(); plt.show()

The same photographs, differently. **No new information** — that is the point and also the limit. Augmentation resamples the manifold you already have; it cannot show the model a breed it has never seen.

## The model, with augmentation and dropout

In [ ]:
inputs = keras.Input(shape=(180, 180, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)
x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
x = layers.Flatten()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs=inputs, outputs=outputs)

model.compile(loss="binary_crossentropy", optimizer="rmsprop",
              metrics=["accuracy"])

validation_dataset = image_dataset_from_directory(
    new_base_dir / "validation", image_size=(180, 180), batch_size=32)

callbacks = [keras.callbacks.ModelCheckpoint(
    filepath="convnet_with_augmentation.keras",
    save_best_only=True, monitor="val_loss")]
history = model.fit(train_dataset, epochs=100,
                    validation_data=validation_dataset,
                    callbacks=callbacks, verbose=2)

> ⚠️ **One hundred epochs, not thirty.** Augmentation slows overfitting down, so the model needs longer to reach its best. Running it for thirty and concluding augmentation *did not help* is a common and expensive mistake.

## The result

In [ ]:
import numpy as np

h = history.history
epochs = range(1, len(h["accuracy"]) + 1)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(epochs, h["accuracy"], lw=1, label="training")
a1.plot(epochs, h["val_accuracy"], lw=1.6, label="validation")
a1.set_title("Accuracy"); a1.legend(); a1.set_xlabel("epoch")
a2.plot(epochs, h["loss"], lw=1, label="training")
a2.plot(epochs, h["val_loss"], lw=1.6, label="validation")
a2.set_title("Loss"); a2.legend(); a2.set_xlabel("epoch")
plt.tight_layout(); plt.show()

from keras.utils import image_dataset_from_directory
test_dataset = image_dataset_from_directory(
    new_base_dir / "test", image_size=(180, 180), batch_size=32)
best = keras.models.load_model("convnet_with_augmentation.keras")
print(f"test accuracy: {best.evaluate(test_dataset, verbose=0)[1]:.3f}")
print(f"validation loss bottoms out at epoch "
      f"{int(np.argmin(h['val_loss']))+1} of {len(epochs)}")

Expected output:

```
test accuracy: ~0.83
```

## Where the ten points came from, and why not more

| | test accuracy |
|---|---|
| from scratch, no augmentation | ~0.70 |
| + augmentation + dropout | **~0.83** |
| pretrained backbone (notebook 04) | ~0.97 |

Augmentation is worth thirteen points and stops there. It **resamples the manifold you have**; it does not add information the 2,000 photographs never contained.

The pretrained model in notebook 04 gets its advantage from somewhere else entirely: 1.4 million images it was trained on before you arrived.

## Choosing augmentations that are true

In [ ]:
# A horizontal flip of a cat is still a cat.
# A horizontal flip of a digit is not still that digit.
from keras.datasets import mnist
(x, y), _ = mnist.load_data()

flip = layers.RandomFlip("horizontal")
img = x[0:1].reshape(1, 28, 28, 1).astype("float32")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(5, 2.6))
a1.imshow(img[0, :, :, 0], cmap="gray_r"); a1.set_title(f"label {y[0]}"); a1.axis("off")
a2.imshow(flip(img, training=True)[0, :, :, 0], cmap="gray_r")
a2.set_title("flipped -- still a 5?"); a2.axis("off")
plt.show()

**An augmentation encodes a claim about your data**: *this transformation does not change the label*. Horizontal flip is true of animals and false of digits, text, and most medical imaging. Getting it wrong teaches the model something untrue, with no error message.

---

## What to take away

- Augmentation layers are training-only, like dropout, and belong inside the model.
- It delays overfitting, so **train for longer** — judging it at the old epoch count understates it.
- Worth about thirteen points here; it resamples the manifold rather than adding information.
- Every augmentation asserts that a transformation preserves the label. Check that it does.